# ResNet Transfer Learning (real torchvision weights)

Loads a pretrained ResNet18 from torchvision, extracts embeddings, and trains a small classifier on digits-as-proxy dataset.

_Last rebuild: **2026-02-16 06:18:51**_

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
torch.manual_seed(42)


In [2]:
# Use digits dataset as a tiny image dataset (8x8) then upsample to 224x224 RGB
D = load_digits()
X = D.images.astype('float32') / 16.0
Y = D.target

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

def to_rgb224(x):
    # x: (N,8,8)
    t = torch.tensor(x).unsqueeze(1)  # (N,1,8,8)
    t = torch.nn.functional.interpolate(t, size=(224,224), mode='bilinear', align_corners=False)
    t = t.repeat(1,3,1,1)
    return t

Xtr = to_rgb224(X_train)
Xte = to_rgb224(X_test)
Ytr = torch.tensor(y_train, dtype=torch.long)
Yte = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=32, shuffle=True)
Xtr.shape

torch.Size([1437, 3, 224, 224])

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Pretrained resnet18 (downloads weights on first run)
weights = ResNet18_Weights.DEFAULT
backbone = resnet18(weights=weights)
backbone.fc = nn.Identity()  # output embeddings
backbone = backbone.to(device)
backbone.eval()

# simple linear classifier on top (train only head)
head = nn.Linear(512, 10).to(device)
opt = torch.optim.Adam(head.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(3):
    head.train()
    losses=[]
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        with torch.no_grad():
            emb = backbone(xb)
        logits = head(emb)
        loss = loss_fn(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    print('epoch', epoch+1, 'loss', float(np.mean(losses)))

# eval
head.eval()
with torch.no_grad():
    emb = backbone(Xte.to(device))
    pred = head(emb).argmax(dim=1).cpu().numpy()

print('accuracy:', accuracy_score(y_test, pred))

epoch 1 loss 2.091036711798774
epoch 2 loss 1.6162369516160753
epoch 3 loss 1.3190627230538263
accuracy: 0.7638888888888888


In [4]:
print('DONE 2026-02-16 06:18:51')

DONE 2026-02-16 06:18:51
